# CIS 3120 Library Database Project

## Step 1: Import Libraries and file paths

In [1]:
import sqlite3
import csv
import urllib.request
import os

# GitHub raw URLs for the three CSV files
# Replace <username> and <repo> with the actual values provided by your instructor

BASE_URL = "https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/CIS3120-BMWB/refs/heads/main/"

BOOK_URL = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL = BASE_URL + "Loan.csv"

# Local paths in the Colab /content directory
BOOK_PATH = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH = "/content/Loan.csv"

DB_PATH = "/content/library.db"

print("Libraries imported and file paths set.")

Libraries imported and file paths set.


## Step 2 Download CSV Files

In [2]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path}")

# Confirm all three files are found
print("Book.csv found:",   os.path.exists(BOOK_PATH))
print("Member.csv found:", os.path.exists(MEMBER_PATH))
print("Loan.csv found:",   os.path.exists(LOAN_PATH))

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv
Book.csv found: True
Member.csv found: True
Loan.csv found: True


## Step 3: Create Database and tables

Connect to library.db and create the three tables: Book, Member, and Loan

In [3]:
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')

# Create Book table
conn.execute('''
CREATE TABLE IF NOT EXISTS Book (
    callNo  TEXT    NOT NULL,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    PRIMARY KEY (callNo)
);
''')

# Create Member table
conn.execute('''
CREATE TABLE IF NOT EXISTS Member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
);
''')

# Create Loan table
conn.execute('''
CREATE TABLE IF NOT EXISTS Loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id)     REFERENCES Member(id)
);
''')

conn.commit()
print("Tables created.")

Tables created.


## Step 4: Load Book.csv into the book table

In [4]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);',
            (row['callNo'], row['title'], row['author'])
        )

conn.commit()
print('Book rows loaded:', conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0])

Book rows loaded: 11


## Step 5: Load Member.csv into the member table

In [5]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);',
            (int(row['id']), row['firstname'], row['lastName'])
        )

conn.commit()
print('Member rows loaded:', conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0])

Member rows loaded: 4


## Step 6: Load Loan.csv into the loan table


In [6]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Convert empty dateReturned to None (-> NULL in SQLite)
        date_returned = row['dateReturned'] if row['dateReturned'].strip() else None

        conn.execute(
            '''INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
               VALUES (?, ?, ?, ?, ?);''',
            (row['callNo'], int(row['id']),
             row['dateBorrowed'], date_returned, row['dateDue'])
        )

conn.commit()
print('Loan rows loaded:', conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0])

Loan rows loaded: 4


## Query 1: All Books

Question: Retrieve all columns from the Book table, ordered alphabetically by author name.

In [7]:
# Query 1: All books ordered alphabetically by author
cursor = conn.execute('''
    SELECT *
    FROM   Book
    ORDER BY author;
''')

print("All books ordered by author:")
for row in cursor:
    print(row)

All books ordered by author:
('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


## Query 2: Books not yet returned

Question: Retrieve the title of each book and the full name of the member who borrowed it, for all loans where `dateReturned` is NULL (book is still checked out).

In [8]:
# Query 2: Books not yet returned
cursor = conn.execute('''
    SELECT   Book.title,
             Member.firstname,
             Member.lastName
    FROM     Loan
    JOIN     Book   ON Loan.callNo = Book.callNo
    JOIN     Member ON Loan.id     = Member.id
    WHERE    Loan.dateReturned IS NULL;
''')

print("Books not yet returned:")
for row in cursor:
    print(row)

Books not yet returned:
("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


## Query 3: Loan history for a specific book

Question: Retrieve the full loan history for the book with callNo = 'R 141 E45 2006', showing the member's full name, dateBorrowed, dateDue, and dateReturned, ordered by dateBorrowed.

In [9]:
# Query 3 : Loan history for book with callNo 'R 141 E45 2006'
cursor = conn.execute('''
    SELECT   Member.firstname,
             Member.lastName,
             Loan.dateBorrowed,
             Loan.dateDue,
             Loan.dateReturned
    FROM     Loan
    JOIN     Member ON Loan.id = Member.id
    WHERE    Loan.callNo = 'R 141 E45 2006'
    ORDER BY Loan.dateBorrowed ASC;
''')

print("Loan history for R 141 E45 2006:")
for row in cursor:
    print(row)

Loan history for R 141 E45 2006:
('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


## Query 4: Members who have never borrowed a book

Question: Retrieve the id, firstname, and lastName of every member who has no loan record at all in the Loan table.

In [10]:
# Query 4: Members who have never borrowed a book
cursor = conn.execute('''
    SELECT   Member.id,
             Member.firstname,
             Member.lastName
    FROM     Member
    LEFT JOIN Loan ON Member.id = Loan.id
    WHERE    Loan.id IS NULL;
''')

print("Members who have never borrowed a book:")
for row in cursor:
    print(row)

Members who have never borrowed a book:
(4, 'John', 'Martin')


## Query 5: Count of loans per member

Question: Retrieve each member's full name and the total number of loans they have made, including members with zero loans.

In [11]:
# Query 5: Count of loans per member including zero loans
cursor = conn.execute('''
    SELECT   Member.firstname,
             Member.lastName,
             COUNT(Loan.callNo) AS loan_count
    FROM     Member
    LEFT JOIN Loan ON Member.id = Loan.id
    GROUP BY Member.id,
             Member.firstname,
             Member.lastName
    ORDER BY loan_count DESC;
''')

print("Number of loans per member:")
for row in cursor:
    print(row)

Number of loans per member:
('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


## Query 6: Free Choice Analytical Query

Question: Which books were returned on time dateReturned on or before dateDue?

Why it is useful? A library needs to track which members return books on time to identify reliable borrowers and enforce late fee policies when needed. This query joins all three tables and uses a WHERE condition not used in any previous query.

In [12]:
# Query 6: Books returned on time
cursor = conn.execute('''
    SELECT   Book.title,
             Member.firstname,
             Member.lastName,
             Loan.dateDue,
             Loan.dateReturned
    FROM     Loan
    JOIN     Book   ON Loan.callNo = Book.callNo
    JOIN     Member ON Loan.id     = Member.id
    WHERE    Loan.dateReturned IS NOT NULL
    AND      Loan.dateReturned <= Loan.dateDue;
''')

print("Books returned on time:")
for row in cursor:
    print(row)

Books returned on time:
('Medieval medicine and the plague', 'Betty', 'Freeman', '4/15/2014 0:00', '4/15/2014 0:00')


## Summary:



# One data quality issue I noticed is that the dateReturned column in Loan.csv stores empty strings instead of NULL for books that have not been returned yet. I converted those empty fields in python so that SQL saves them as NULL. The queries would not be able to find the missing return dates correctly. Ultimately, this dataset is very small it only has 11 books, 4 members, and 4 loans, like a real library would have thousands of records. It is also missing information, for example, how many copies exist and the member's contact information, etc.